In [1]:
version = "REPLACE_PACKAGE_VERSION"

# Assignment 2: Mining Itemsets (Part III)

## Apriori Algorithm under the Hood

In part III of the assignment, we are going to continue our exploration in mining frequent itemsets. Specifically, we are going to examine a few key steps in the Apriori algorithm.

First, let's import the packages and dependencies that will be used in this part of the assignment. .

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
from mads.lib.path import assets

**<span style="color:red">NOTE: These are all the imports we need to make for this assignment (Part III). You should not make other imports in your submitted notebook. You will receive 0 points for the exercises if your solution includes additional imports.</span>**

##### A critical step of the Apriori algorithm is *candidate generation*. That is, candidate *(k+1)*-itemsets should be generated from frequent *k*-itemsets. In the following exercise, we want you to generate candidate 3-itemsets based on the frequent 2-itemsets.

### Exercise 3 (25 pts)

**Candidate Generation** (20 pts)

You will need to **correct** the `generate_candidate_3_itemsets` function, which takes in a list of frequent 2-itemsets and returns a list of the candidate 3-itemsets ("candidate" means that they may or may not be frequent). Please represent an itemset as a  `set` in Python. Make sure that for each candidate 3-itemset in your returned list, **at least one** of its size-2 subset is a frequent 2-itemset, and your list does not contain duplicated itemsets.

We have prepared the frequent 2-itemsets for you, which we will help you load from the file named `product_frequent_2_itemsets.csv`. We will evaluate your function by feeding in the loaded 2-itemsets and examining the return value. Note that the loaded 2-itemsets are different from what you will get with the `product_frequent_itemsets` function you implemented in Part II, as we have eliminated some products.

You will receive 20 points if
1. every candidate 3-itemset in your returned list is a **superset of at least
   one** frequent 2-itemset, and 
2. every 3-itemset that has a frequent size-2 subset is already in your list,
   and your list does not contain duplicated sets. 

Note that this pruning procedure won't give us the smallest set of candidates. In the following challenge exercise, we will further prune the candidate itemsets.

**Challenge Exercise** (5 pts)

We can further prune the candidate itemsets by requiring **all** size-2 subsets of each candidate  3-itemset to be frequent. 

Think about the example you've seen in the lecture. Suppose {🍺, 🍼}, {🍺, 🍋}, {🍼, 🍭}, {🍼, 🍋} are all the frequent 2-itemsets. In the lecture, we said {🍺, 🍼, 🍋}, {🍺, 🍼, 🍭}, {🍼, 🍭, 🍋} are candidate 3-itemsets because each 3-itemset is a superset of **at least one** frequent 2-itemset. However, a larger itemset can never be frequent as long as one of its subset is not frequent. In this case, {🍺, 🍼, 🍭} can never been frequent because {🍺, 🍭} is not frequent. Neither can {🍼, 🍭, 🍋}, as the subset {🍭, 🍋} is not frequent. Ideally, we should be able to exclude the two candidate 3-itemsets {🍺, 🍼, 🍭} and {🍼, 🍭, 🍋} even without scanning the database for counting.

For this challenge exercise, please **prune your generated candidate 3-itemsets by requiring all their subsets to be frequent**. You will receive 5 points if the pruning is done correctly.

**Hint**: There are two errors in the function. One of them is related to the
challenge exercise. i.e., under `if pruning:`.

In [7]:
def generate_candidate_3_itemsets(base_itemsets, pruning=False):
    """
    Correct this function. There are at least two bugs in this function.
    The bugs are related to the logic of the function. You may assume the
    syntax and the structure of the function is correct.
    """
    # create an empty list for candidates
    candidates = []

    # collect all unique items that appear in the frequent 2-itemsets
    candidate_items = set()
    for itemset in base_itemsets:
        for i in itemset:
            candidate_items.add(i)

    # loop through each frequent 2-itemsets
    for itemset in base_itemsets:
        pair = list(itemset)
        # try adding one new item to the pair
        for item in candidate_items:
            if item not in pair:  # prevents us from adding an item that is already in the pair
                # w/o this condition, we would not actually create a new 3-itemset
                target = set(pair + [item])
                # avoid duplicate candidates
                if target not in candidates:
                    # optional apriori pruning
                    if pruning:
                        if {pair[0], item} in base_itemsets and \
                            {pair[1], item} in base_itemsets:
                            candidates.append(target)
                    else:
                        candidates.append(target)

    # return the final candidate list
    return candidates

*What if you need to implement the function `generate_candidate_n_itemsets`?
Can you come up with an elegant solution?*

In [8]:
file_itemsets = assets.find("prod_frequent_2_itemsets.csv")
frequent_2_itemsets = []
with open(file_itemsets, 'r') as fin:
    frequent_2_itemsets.extend({int(x)
                                for x in line.strip().split(",")}
                               for line in fin)

candidate_3_itemsets = generate_candidate_3_itemsets(frequent_2_itemsets)

In [9]:
# This code block tests whether the `generate_candidate_3_itemsets` function is implemented correctly.
# We hide some tests, so passing all the displayed assertions does not guarantee full points.

# obtain return value
candidate_3_itemsets = generate_candidate_3_itemsets(frequent_2_itemsets,
                                                     pruning=False)

# assertions
examined = []
for candidate in candidate_3_itemsets:
    assert len(
        candidate) == 3, f"[Exercise 3] Candidate {candidate} not a 3-itemset"
    assert candidate not in examined, f"[Exercise 3] Duplicate 3-itemsets: {candidate}"
    c1, c2, c3 = list(candidate)
    assert {c1, c2} in frequent_2_itemsets or {
        c1, c3
    } in frequent_2_itemsets or {
        c2, c3
    } in frequent_2_itemsets, f"[Exercise 3] Candidate {candidate} does not contain frequent 2-itemsets"

    examined.append(candidate)


In [10]:
# This code block tests whether the `generate_candidate_3_itemsets` function can perform the
# additional pruning for the challenge exercise.
# We hide some tests, so passing all the displayed assertions does not guarantee full points.

# obtain return value
candidate_3_itemsets = generate_candidate_3_itemsets(frequent_2_itemsets,
                                                     pruning=True)

# assertions
examined = []
for candidate in candidate_3_itemsets:
    assert len(
        candidate) == 3, f"[Exercise 3] Candidate {candidate} not a 3-itemset"
    assert candidate not in examined, f"[Exercise 3] Duplicate 3-itemsets: {candidate}"
    c1, c2, c3 = list(candidate)
    assert {c1, c2} in frequent_2_itemsets or {
        c1, c3
    } in frequent_2_itemsets or {
        c2, c3
    } in frequent_2_itemsets, f"[Exercise 3] Candidate {candidate} does not contain frequent 2-itemsets"

    examined.append(candidate)


Before heading to the next exercise, please run the following code block to load the data set prepared in Part I of this assignment.

In [11]:
# Load data from files.
file_orders = assets.find("orders.csv.zip")
file_products = assets.find("products.csv.zip")
orders = pd.read_csv(file_orders, nrows=10000)
products = pd.read_csv(file_products)

# Group orders by order id and merge them into a set.
order_baskets = orders.groupby("order_id")["product_id"].apply(frozenset)

# Convert the above pandas Series to a pandas DataFrame.
order_baskets = order_baskets.to_frame(name="products_id")

# Create the name map for later reference.
product_name_map = dict(zip(products["product_id"], products["product_name"]))

##### Exercise 4 (15 pts)

With the candidate itemsets ready, the final step in one iteration of the Apriori algorithm is to scan the database (in our case, you can either scan the `order_baskets`) and count the occurrence of each candidate itemset, divide it by the total number of records to derive the support, and output candidate itemsets whose support meets the threshold. 

Please complete the `calculate_frequent_itemsets` function below. The input
(`candidate_itemsets`) is a list of candidate 3-itemsets. Your function should
return a complete list of frequent 3-itemsets with a minimal support of
`min_support`. The returned list should not contain duplicated itemsets. The
order of the list does not matter.

**Hint**: You may want to use the `set.issubset()` method to check if a candidate
itemset is a subset of an order basket.

In [14]:
def calculate_frequent_itemsets(candidate_itemsets, min_support=0.005):

    # initialize storage and total basket count
    frequent_itemsets = []
    num_baskets = len(order_baskets)

    # loop through each candidate itemset
    for candidate in candidate_itemsets:
        count = 0

        # check whether the candidate appears in each basket
        for basket in order_baskets["products_id"]:
            if candidate.issubset(basket):
                count += 1

        # compute support
        support = count / num_baskets

        # keep only itemsets meeting the threshold
        if support >= min_support and candidate not in frequent_itemsets:
            frequent_itemsets.append(candidate)

    # return the frequent itemsets
    return frequent_itemsets

In [15]:
calculate_frequent_itemsets(candidate_3_itemsets)

[{21903, 24852, 25890}, {21903, 24852, 45007}, {13176, 21137, 47209}]

In [16]:
# This code block tests whether the `calculate_frequent_itemsets` function work as expected.
# We hide some tests, so passing all the displayed assertions does not guarantee full points.

answer = calculate_frequent_itemsets(candidate_3_itemsets)

assert len(answer) == 3
assert {21903, 24852, 45007} in answer
